# ***New Code***

# ***Cell 1 — install + imports***

In [16]:
!pip -q install -U imbalanced-learn

import re, time
import numpy as np
import pandas as pd

from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2, f_classif, VarianceThreshold
from sklearn.decomposition import PCA, FastICA, TruncatedSVD
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

np.random.seed(42)


# ***Cell 2 — load dataset (from Drive path OR uploaded path)***

In [18]:
# Option A: from Drive
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = "/content/drive/MyDrive/IOT/Dataset/windows7_dataset.csv"

# Option B: uploaded to Colab session
#DATA_PATH = "/content/windows7_dataset.csv"

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Loaded:", df.shape)
df.head()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (28367, 135)


,ts,Processor(_Total) DPC Rate,Processor(_Total) pct_ Idle Time,Processor(_Total) pct_ C3 Time,Processor(_Total) pct_ Interrupt Time,Processor(_Total) pct_ C2 Time,Processor(_Total) pct_ User Time,Processor(_Total) pct_ C1 Time,Processor(_Total) pct_ Processor Time,Processor(_Total) C1 Transitions sec,...,Memory Available MBytes,Memory Modified Page List Bytes,Memory Cache Faults sec,Memory Committed Bytes,Memory System Driver Total Bytes,Memory Pages Input sec,Memory Pool Paged Resident Bytes,Memory Write Copies sec,label,type
0,1554185566,0,90.20833333,0,0.208333333,0,2.083333333,88.00928867,9.791666667,216.6151809,...,414,26374144,31.95524227,926330880,5849088,1957.675542,48304128,0,0,normal
1,1554185581,0,99.79166667,0,0,0,0.104166667,98.92048733,0.208333333,67.86145672,...,418,27045888,1.199907879,924794880,5849088,0.133323098,48107520,0,0,normal
2,1554185596,0,99.79166667,0,0,0,0.208333333,99.17886667,0.208333333,65.9999186,...,419,27353088,0.133333169,925319168,5849088,0,48033792,0,0,normal
3,1554185611,0,99.6875,0,0,0,0.104166667,99.218316,0.3125,67.39927613,...,425,27533312,0.133331901,916725760,5849088,1.066655211,47960064,0.133331901,0,normal
4,1554185626,0,99.79166667,0,0,0,0.208333333,99.24871267,0.208333333,65.53277281,...,426,27688960,0.133332193,916025344,5849088,0,47943680,0,0,normal


# ***Cell 3 — clean column names + choose target safely (NO target corruption)***

In [19]:
def clean_column_name(col: str) -> str:
    col = re.sub(r"[^A-Za-z0-9_]", "_", str(col))
    col = re.sub(r"_+", "_", col).strip("_")
    return col

df = df.copy()
df.columns = [clean_column_name(c) for c in df.columns]

# Choose ONE target:
TARGET_COL = "type"     # multiclass
# TARGET_COL = "label"  # binary

assert TARGET_COL in df.columns, f"{TARGET_COL} not found. Columns end with: {df.columns[-10:].tolist()}"

# Prevent leakage: remove the other label column from features
leak_cols = [c for c in ["label", "type"] if c in df.columns and c != TARGET_COL]

y_raw = df[TARGET_COL].copy()
X = df.drop(columns=[TARGET_COL] + leak_cols).copy()

# Optional: drop timestamp feature (usually not meaningful for IDS modeling)
if "ts" in X.columns:
    X = X.drop(columns=["ts"])

# Convert only FEATURE object cols to numeric
for col in X.columns:
    if X[col].dtype == "object":
        s = (X[col].astype(str)
             .str.replace("%", "", regex=False)
             .str.replace(",", "", regex=False)
             .str.strip()
             .replace(["nan", "None", "?", ""], np.nan))
        X[col] = pd.to_numeric(s, errors="coerce")

X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

# Drop constant feature columns
nunique = X.nunique(dropna=False)
X = X.drop(columns=nunique[nunique <= 1].index.tolist())

# Encode y
le = LabelEncoder()
y = le.fit_transform(y_raw.astype(str))

print("X:", X.shape, "| classes:", len(le.classes_))
print("Class counts:", Counter(y))
print("Class names:", list(le.classes_))


X: (28367, 78) | classes: 8
Class counts: Counter({np.int64(3): 22387, np.int64(1): 2134, np.int64(0): 1779, np.int64(2): 998, np.int64(4): 757, np.int64(6): 226, np.int64(5): 82, np.int64(7): 4})
Class names: ['backdoor', 'ddos', 'injection', 'normal', 'password', 'ransomware', 'scanning', 'xss']


# ***Cell 4 — split + RandomUnderSampler strategy (critical fix for the dataset)***

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

train_counts = Counter(y_train)
n_classes = len(train_counts)

# Proposed method uses RandomUnderSampler. The key is sampling_strategy.
if n_classes == 2:
    # Safe for binary: balance majority down to minority
    sampler = RandomUnderSampler(random_state=42)
else:
    # Multiclass: DO NOT downsample to the absolute smallest class (xss=4)
    # Cap large classes to the MEDIAN class size (still under-sampling, but not catastrophic).
    target_n = int(np.median(list(train_counts.values())))
    sampling_strategy = {cls: min(cnt, target_n) for cls, cnt in train_counts.items()}
    sampler = RandomUnderSampler(random_state=42, sampling_strategy=sampling_strategy)

print("Train before:", train_counts)
Xr, yr = sampler.fit_resample(X_train, y_train)
print("Train after: ", Counter(yr))
print("Train size:", len(y_train), "->", len(yr))


Train before: Counter({np.int64(3): 17909, np.int64(1): 1707, np.int64(0): 1423, np.int64(2): 798, np.int64(4): 606, np.int64(6): 181, np.int64(5): 66, np.int64(7): 3})
Train after:  Counter({np.int64(0): 702, np.int64(1): 702, np.int64(2): 702, np.int64(3): 702, np.int64(4): 606, np.int64(6): 181, np.int64(5): 66, np.int64(7): 3})
Train size: 22693 -> 3664


# ***Cell 5 — define FS/FE/classifiers + run full grid***

In [21]:
def make_selector(name, k):
    if name == "VT":
        return VarianceThreshold(threshold=0.0)
    if name == "ANOVA":
        return SelectKBest(score_func=f_classif, k=k)
    if name == "Chi2":
        return SelectKBest(score_func=chi2, k=k)
    raise ValueError(name)

def make_extractor(name, n_components, n_classes):
    if name == "PCA":
        return PCA(n_components=n_components, random_state=42)
    if name == "SVD":
        return TruncatedSVD(n_components=n_components, random_state=42)
    if name == "ICA":
        return FastICA(n_components=n_components, random_state=42, max_iter=1000, tol=0.01)
    if name == "LDA":
        lda_nc = min(n_components, max(1, n_classes - 1))
        return LinearDiscriminantAnalysis(n_components=lda_nc)
    raise ValueError(name)

CLASSIFIERS = {
    "DT":  DecisionTreeClassifier(random_state=42),
    "LR":  LogisticRegression(max_iter=5000, solver="saga"),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "NB":  GaussianNB(),
    "SVM": SVC(kernel="rbf", gamma="scale"),
}

FS_LIST = ["VT", "ANOVA", "Chi2"]
FE_LIST = ["PCA", "LDA", "ICA", "SVD"]

# k + components (deterministic defaults)
k = min(30, X_train.shape[1])
n_components = min(20, max(2, k))

def metrics_all(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
        "macro_precision": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
    }

results = []
ncls = len(np.unique(y_train))

for fs_name in FS_LIST:
    for fe_name in FE_LIST:
        for clf_name, clf in CLASSIFIERS.items():

            selector = make_selector(fs_name, k=k)
            extractor = make_extractor(fe_name, n_components=n_components, n_classes=ncls)

            # Chi2 needs non-negative => MinMaxScaler
            scaler = MinMaxScaler() if fs_name == "Chi2" else StandardScaler()

            # VT must be BEFORE scaler (otherwise scaler makes variances ~1)
            if fs_name == "VT":
                steps = [("sampler", sampler), ("fs", selector), ("scaler", scaler), ("fe", extractor), ("clf", clf)]
            else:
                steps = [("sampler", sampler), ("scaler", scaler), ("fs", selector), ("fe", extractor), ("clf", clf)]

            pipe = ImbPipeline(steps=steps)

            t0 = time.time()
            try:
                pipe.fit(X_train, y_train)
                y_pred = pipe.predict(X_test)
                m = metrics_all(y_test, y_pred)
                results.append({"FS": fs_name, "FE": fe_name, "CLF": clf_name, **m, "fit_time_s": time.time()-t0})
            except Exception as e:
                results.append({"FS": fs_name, "FE": fe_name, "CLF": clf_name,
                                "accuracy": np.nan, "macro_f1": np.nan, "weighted_f1": np.nan,
                                "macro_precision": np.nan, "macro_recall": np.nan,
                                "fit_time_s": np.nan, "error": str(e)[:200]})

res_df = pd.DataFrame(results)

# Rank by macro_f1 (paper-style fair-to-all-classes); also shows weighted_f1 for realism
res_sorted = res_df.sort_values(["macro_f1", "accuracy"], ascending=False)
print("Top 15:")
display(res_sorted.head(15))


Top 15:


,FS,FE,CLF,accuracy,macro_f1,weighted_f1,macro_precision,macro_recall,fit_time_s
7,VT,LDA,KNN,0.955411,0.794409,0.961095,0.754581,0.865455,0.249412
47,Chi2,LDA,KNN,0.948185,0.788393,0.955091,0.748301,0.861883,0.530888
45,Chi2,LDA,DT,0.943602,0.782407,0.951220,0.749148,0.847889,0.118218
34,ANOVA,ICA,SVM,0.912936,0.769249,0.932485,0.741695,0.852630,0.993478
27,ANOVA,LDA,KNN,0.936553,0.761677,0.946295,0.715284,0.861906,0.198576
32,ANOVA,ICA,KNN,0.911526,0.759784,0.928533,0.719008,0.854154,0.382595
31,ANOVA,ICA,LR,0.918047,0.757437,0.934425,0.723616,0.846046,4.436051
22,ANOVA,PCA,KNN,0.924392,0.756869,0.938832,0.712314,0.853415,0.246967
37,ANOVA,SVD,KNN,0.924392,0.756869,0.938832,0.712314,0.853415,0.308848
57,Chi2,SVD,KNN,0.908178,0.754547,0.925704,0.711825,0.856024,0.337890


# ***Cell 6 — best model report + confusion matrix***

In [22]:
best = res_sorted.dropna(subset=["macro_f1"]).iloc[0]
print("BEST:", best.to_dict())

best_fs, best_fe, best_clf = best["FS"], best["FE"], best["CLF"]

selector = make_selector(best_fs, k=k)
extractor = make_extractor(best_fe, n_components=n_components, n_classes=len(np.unique(y_train)))
scaler = MinMaxScaler() if best_fs == "Chi2" else StandardScaler()
clf = CLASSIFIERS[best_clf]

if best_fs == "VT":
    best_pipe = ImbPipeline([("sampler", sampler), ("fs", selector), ("scaler", scaler), ("fe", extractor), ("clf", clf)])
else:
    best_pipe = ImbPipeline([("sampler", sampler), ("scaler", scaler), ("fs", selector), ("fe", extractor), ("clf", clf)])

best_pipe.fit(X_train, y_train)
y_pred = best_pipe.predict(X_test)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion matrix shape:", cm.shape)
cm


BEST: {'FS': 'VT', 'FE': 'LDA', 'CLF': 'KNN', 'accuracy': 0.9554106450475854, 'macro_f1': 0.7944086577842349, 'weighted_f1': 0.9610952163298393, 'macro_precision': 0.7545811699875441, 'macro_recall': 0.8654545645074009, 'fit_time_s': 0.24941182136535645}

Classification report:
              precision    recall  f1-score   support

    backdoor       0.90      0.99      0.94       356
        ddos       0.98      1.00      0.99       427
   injection       0.99      1.00      1.00       200
      normal       1.00      0.94      0.97      4478
    password       0.46      0.99      0.62       151
  ransomware       1.00      1.00      1.00        16
    scanning       0.71      1.00      0.83        45
         xss       0.00      0.00      0.00         1

    accuracy                           0.96      5674
   macro avg       0.75      0.87      0.79      5674
weighted avg       0.97      0.96      0.96      5674


Confusion matrix shape: (8, 8)


array([[ 354,    0,    0,    2,    0,    0,    0,    0],
       [   0,  426,    0,    1,    0,    0,    0,    0],
       [   0,    0,  200,    0,    0,    0,    0,    0],
       [  40,    9,    2, 4231,  177,    0,   18,    1],
       [   0,    0,    0,    2,  149,    0,    0,    0],
       [   0,    0,    0,    0,    0,   16,    0,    0],
       [   0,    0,    0,    0,    0,    0,   45,    0],
       [   0,    0,    0,    0,    1,    0,    0,    0]])

# ***Cell 7 — save results***

In [ ]:
OUT_PATH = "/content/drive/MyDrive/IOT/Results/proposed_method_results_windows7.csv"
res_sorted.to_csv(OUT_PATH, index=False)
print("Saved:", OUT_PATH)
